In [1]:

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

Start a Spark Session


In [2]:
spark = SparkSession.builder \
    .appName("world population Linear Regression") \
    .master("local[*]") \
    .getOrCreate()

print("Spark is ready!")
print("Spark version:", spark.version)

Spark is ready!
Spark version: 4.0.3


Download the Dataset

In [3]:
df = spark.read.csv('/content/world_population_by_country_2026.csv', header=True, inferSchema=True)
df.show(5)

+----+-------------+---------------+-------------+-----------+---------------+-------------+------------+--------------+----------+--------------------+---------------+
|Rank|      Country|Population_2026|Yearly_Change| Net_Change|Density_per_km2|Land_Area_km2|Migrants_Net|Fertility_Rate|Median_Age|Urban_Population_pct|World_Share_pct|
+----+-------------+---------------+-------------+-----------+---------------+-------------+------------+--------------+----------+--------------------+---------------+
|   1|        India|     1476625576|         0.87|1.2760051E7|            497|      2973190|   -440456.0|          1.93|      29.2|                37.6|          17.79|
|   2|        China|     1412914089|        -0.22| -3182005.0|            150|      9388211|   -232107.0|          1.03|      40.6|                68.7|          17.02|
|   3|United States|      349035494|         0.51|  1759687.0|             38|      9147420|   1177848.0|          1.63|      38.7|                83.1|   

Basic Exploratory Data Analysis

In [4]:
df.printSchema()

print("Number of records:", df.count())

root
 |-- Rank: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- Population_2026: integer (nullable = true)
 |-- Yearly_Change: double (nullable = true)
 |-- Net_Change: double (nullable = true)
 |-- Density_per_km2: integer (nullable = true)
 |-- Land_Area_km2: integer (nullable = true)
 |-- Migrants_Net: double (nullable = true)
 |-- Fertility_Rate: double (nullable = true)
 |-- Median_Age: double (nullable = true)
 |-- Urban_Population_pct: double (nullable = true)
 |-- World_Share_pct: double (nullable = true)

Number of records: 233


In [5]:
# Display summary statistics for numerical columns
df.describe().show()

+-------+----------------+-----------+--------------------+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+--------------------+-------------------+
|summary|            Rank|    Country|     Population_2026|     Yearly_Change|        Net_Change|  Density_per_km2|     Land_Area_km2|     Migrants_Net|    Fertility_Rate|        Median_Age|Urban_Population_pct|    World_Share_pct|
+-------+----------------+-----------+--------------------+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+--------------------+-------------------+
|  count|             233|        233|                 233|               233|               233|              233|               233|              233|               233|               233|                 233|                233|
|   mean|           117.0|       NULL| 3.561724801716738E7|0.83600858369

In [6]:
# Check for missing values
from pyspark.sql.functions import col, sum

df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+----+-------+---------------+-------------+----------+---------------+-------------+------------+--------------+----------+--------------------+---------------+
|Rank|Country|Population_2026|Yearly_Change|Net_Change|Density_per_km2|Land_Area_km2|Migrants_Net|Fertility_Rate|Median_Age|Urban_Population_pct|World_Share_pct|
+----+-------+---------------+-------------+----------+---------------+-------------+------------+--------------+----------+--------------------+---------------+
|   0|      0|              0|            0|         0|              0|            0|           0|             0|         0|                   0|              0|
+----+-------+---------------+-------------+----------+---------------+-------------+------------+--------------+----------+--------------------+---------------+



Prepare Features Using VectorAssembler

In [7]:
feature_cols = [
    'Yearly_Change',
    'Net_Change',
    'Density_per_km2',
    'Land_Area_km2',
    'Migrants_Net',
    'Fertility_Rate',
    'Median_Age',
    'Urban_Population_pct',
    'World_Share_pct'
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

# Transform the DataFrame to include the features vector
df_features = assembler.transform(df)

df_features.select('Country', 'Population_2026', 'features').show(5, truncate=False)

+-------------+---------------+-----------------------------------------------------------------+
|Country      |Population_2026|features                                                         |
+-------------+---------------+-----------------------------------------------------------------+
|India        |1476625576     |[0.87,1.2760051E7,497.0,2973190.0,-440456.0,1.93,29.2,37.6,17.79]|
|China        |1412914089     |[-0.22,-3182005.0,150.0,9388211.0,-232107.0,1.03,40.6,68.7,17.02]|
|United States|349035494      |[0.51,1759687.0,38.0,9147420.0,1177848.0,1.63,38.7,83.1,4.2]     |
|Indonesia    |287886782      |[0.76,2165546.0,159.0,1811570.0,-39472.0,2.08,30.7,60.3,3.47]    |
|Pakistan     |259299791      |[1.6,4080237.0,336.0,770880.0,-1144738.0,3.44,20.8,34.7,3.12]    |
+-------------+---------------+-----------------------------------------------------------------+
only showing top 5 rows


Select the Columns Needed for Machine Learning

In [8]:
# Select the features and label columns
model_df = df_features.select('features', col('Population_2026').alias('label'))
model_df.show(5, truncate=False)

+-----------------------------------------------------------------+----------+
|features                                                         |label     |
+-----------------------------------------------------------------+----------+
|[0.87,1.2760051E7,497.0,2973190.0,-440456.0,1.93,29.2,37.6,17.79]|1476625576|
|[-0.22,-3182005.0,150.0,9388211.0,-232107.0,1.03,40.6,68.7,17.02]|1412914089|
|[0.51,1759687.0,38.0,9147420.0,1177848.0,1.63,38.7,83.1,4.2]     |349035494 |
|[0.76,2165546.0,159.0,1811570.0,-39472.0,2.08,30.7,60.3,3.47]    |287886782 |
|[1.6,4080237.0,336.0,770880.0,-1144738.0,3.44,20.8,34.7,3.12]    |259299791 |
+-----------------------------------------------------------------+----------+
only showing top 5 rows


Split the data into training and testing sets

In [9]:
(trainingData, testData) = model_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training Data Count: {trainingData.count()}")
print(f"Test Data Count: {testData.count()}")

Training Data Count: 198
Test Data Count: 35


### Train a Linear Regression Model

In [10]:
lr = LinearRegression(featuresCol='features', labelCol='label')

lr_model = lr.fit(trainingData)

print("Linear Regression model trained!")

Linear Regression model trained!


Examine the Regression Equation

In [11]:
print("Coefficients:", lr_model.coefficients)
print("Intercept:", lr_model.intercept)

print("\nInflation coefficient:", lr_model.coefficients[0])
print("Unemployment coefficient:", lr_model.coefficients[1])

Coefficients: [8994.388300466853,0.029229272890893165,0.6242290825321896,0.004442229768575204,0.06285193373028035,10616.76158247008,402.4380184248941,-106.72973304353191,82991701.44585848]
Intercept: -42096.633776472045

Inflation coefficient: 8994.388300466853
Unemployment coefficient: 0.029229272890893165


In [13]:
# Make predictions on the test data
predictions = lr_model.transform(testData)

# Select example predictions to show
predictions.select("label", "prediction", "features").show(5)

+-------+------------------+--------------------+
|  label|        prediction|            features|
+-------+------------------+--------------------+
|  45319|25036.333407749953|[-1.54,-710.0,227...|
|2797338| 2790973.381682904|[-1.16,-32806.0,4...|
| 626233| 598622.0634415514|[-1.03,-6496.0,47...|
|3114242|3131628.3768166485|[-0.82,-25853.0,6...|
|  11113|-7449.725770127196|[-0.72,-81.0,79.0...|
+-------+------------------+--------------------+
only showing top 5 rows


In [14]:
# Evaluate the model
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on test data = {rmse}")

evaluator_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)
print(f"R-squared (R2) on test data = {r2}")

Root Mean Squared Error (RMSE) on test data = 165177.1710146403
R-squared (R2) on test data = 0.9999995045800285
